# Lab 2 — Ciclo de treino, vazamento e métricas
**João Rietra · Turma BD-P4-2026.2 · Individual · 1,0 ponto na AV1**

Este notebook responde ao enunciado do Lab 2 com a base Red Wine Quality. Ele é **explicativo**: o código avaliado mora no pacote `lab02lib/` (ao lado deste arquivo) e as células chamam essas funções com poucas linhas. O que importa aqui é o porquê de cada passo e o que cada saída mostra.

Os números citados nos textos depois de cada resultado são gerados pelo código a partir dos arquivos de `resultados/`. Assim o texto não contradiz a saída quando o notebook é executado de novo.

Ordem: preparação → B2 → B1 → A1–A4 → C → D7 → D4 → D5/D6 → D1–D3 → D8 → conferência.

## 0. Ambiente e pasta
Abra este notebook dentro da pasta `lab-02/`, onde estão `apoio.py`, `dados/` e `lab02lib/`. CPU é suficiente; o dispositivo é fixo em `'cpu'`.

In [ ]:
from pathlib import Path
import sys, copy, time, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
from IPython.display import display, Markdown
from apoio import FEATURES, carregar_base, fixar_semente, criar_modelo
from apoio import treinar_epoca, avaliar_perda, obter_escores
from lab02lib import MeuDataset, particionar_sem_vazar, treinar_com_parada

SEED = 42
fixar_semente(SEED)
DISPOSITIVO = 'cpu'
RESULTADOS = Path('resultados')
RESULTADOS.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0])
for pacote in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'torch']:
    print(pacote, importlib.metadata.version(pacote))

### Leitura e conferência da base
`carregar_base` lê o CSV com `;`, confere sua integridade e cria o alvo e os identificadores de controle. As únicas entradas permitidas estão em `FEATURES`. Não aprende média, desvio, pesos ou partições.

Ver as contagens do arquivo não é treinar um modelo. Depois da B2, o teste principal fica fora das decisões e dos experimentos de erro.

In [ ]:
base = carregar_base()
print(base.head().to_string(index=False))
print('\nLinhas:', len(base), '| características:', len(FEATURES))
print('Classe positiva:', int(base.alvo.sum()), '| proporção:', base.alvo.mean())
print('Perfis distintos:', base.grupo_perfil.nunique())
assert set(FEATURES).isdisjoint({'quality', 'alvo', 'id_linha', 'grupo_perfil'})

### Exemplo mínimo: o que faz uma atualização
O bloco a seguir usa **quatro observações artificiais com duas entradas**, apenas para conferir instalação e formas. Não usa Wine Quality, não mede generalização e não atende às partes avaliadas. Um logit é transformado em escore por sigmoide só ao obter previsões; a perda recebe o logit.

In [ ]:
x_exemplo = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_exemplo = torch.tensor([0., 0., 0., 1.])
exemplo_loader = DataLoader(TensorDataset(x_exemplo, y_exemplo), batch_size=2)
exemplo_modelo = criar_modelo(2).to(DISPOSITIVO)
exemplo_otimizador = torch.optim.Adam(exemplo_modelo.parameters(), lr=0.001)
exemplo_perda = treinar_epoca(exemplo_modelo, exemplo_loader, exemplo_otimizador,
                              nn.BCEWithLogitsLoss(), DISPOSITIVO)
print('Teste de ambiente concluído. Perda de treino artificial:', exemplo_perda)
del exemplo_modelo, exemplo_loader, exemplo_otimizador
fixar_semente(SEED)

### Mapa do código
Cada implementação avaliada está em um arquivo do pacote `lab02lib/`. O `apoio.py` do professor não foi modificado.

| Item do enunciado | O que implementa | Arquivo |
|---|---|---|
| B2 | `particionar_sem_vazar`, `auditar_particoes` | [lab02lib/split.py](lab02lib/split.py) |
| A1, A2 | `MeuDataset`, `criar_datasets`, `criar_loaders`, verificações | [lab02lib/data.py](lab02lib/data.py) |
| A3 | `medir_tempos`, `resumir_tempos`, `grafico_tempos` | [lab02lib/timing.py](lab02lib/timing.py) |
| A4 | `demonstrar_batchnorm` | [lab02lib/batchnorm.py](lab02lib/batchnorm.py) |
| C (formas 1 e 4) | experimentos incorreto × corrigido, 5 sementes | [lab02lib/leakage.py](lab02lib/leakage.py) |
| D7 | `treinar_com_parada`, prova de restauração | [lab02lib/train.py](lab02lib/train.py) |
| D4, D5, D6 | custos, busca de limiar, fórmula, calibração, congelamento | [lab02lib/decision.py](lab02lib/decision.py) |
| D1, D2, D3 | métricas, baseline, curvas ROC e PR | [lab02lib/evaluate.py](lab02lib/evaluate.py) |
| MLflow (registro opcional) | rastreio dos experimentos em `mlflow.db` | [lab02lib/tracking.py](lab02lib/tracking.py) |

## B2. Separe antes de normalizar
**Pergunta de uso:** selecionar amostras de vinho tinto de nota alta usando só as 11 medidas físico-químicas.

**Estratégia:** `GroupShuffleSplit` por `grupo_perfil`, duas vezes. A primeira reserva 20% dos grupos para o teste. A segunda tira 25% dos grupos que sobraram para a validação (0,25 de 0,8 = 0,2 do total). Perfis idênticos ficam sempre no mesmo conjunto. A semente 42 foi declarada antes de qualquer modelo e não foi procurada por métrica.

A auditoria confere: contagens, cobertura das 1599 linhas, `id_linha` sem sobreposição, zero perfis compartilhados entre cada par e as duas classes em cada conjunto. Se qualquer regra falhar, a função levanta erro.

In [ ]:
from lab02lib.split import auditar_particoes

particoes = particionar_sem_vazar(base, SEED)
b2_tabela, b2_pares = auditar_particoes(particoes, base)
b2_tabela.to_csv(RESULTADOS / 'b2_auditoria.csv', index=False)
b2_pares.to_csv(RESULTADOS / 'b2_pares.csv', index=False)
display(b2_tabela.round(4))
display(b2_pares)
print('OK: cobertura total das', len(base), 'linhas; id_linha disjunto; 0 perfis compartilhados; as duas classes em cada conjunto.')

In [ ]:
t = b2_tabela.set_index('conjunto')
maior = t.desvio_pp_vs_alvo.abs().max()
display(Markdown(f'''**Como ler.** Treino, validação e teste ficaram com {t.linhas['treino']}, {t.linhas['validacao']} e {t.linhas['teste']} linhas ({100*t.proporcao_linhas['treino']:.1f}%, {100*t.proporcao_linhas['validacao']:.1f}% e {100*t.proporcao_linhas['teste']:.1f}%). O sorteio é por perfil, então as proporções de linhas só se aproximam de 60/20/20. O maior desvio é de {maior:.2f} ponto percentual, {'dentro' if maior <= 3 else 'fora'} da faixa de ±3.

A proporção de positivos não é a mesma nos três conjuntos: {100*t.proporcao_positiva['treino']:.2f}% no treino, {100*t.proporcao_positiva['validacao']:.2f}% na validação ({t.positivos['validacao']} positivos) e {100*t.proporcao_positiva['teste']:.2f}% no teste ({t.positivos['teste']} positivos). Isso vem do sorteio por grupos e eu não mexi na semente para corrigir. A validação tem poucos positivos; vou lembrar disso na escolha do limiar.

Os zeros de perfis compartilhados dizem que nenhuma medida idêntica atravessa a fronteira. Eles não provam que dois vinhos de perfis diferentes são físicos diferentes, nem o contrário.'''))

### B1. Cinco cenários
As respostas dos cinco cenários (avaliações de produtos, turbinas, spam, radiografias e transações) estão no `README.md`, na seção B1, como pede o enunciado. Em todas eu declaro o uso futuro que estou pressupondo e as hipóteses que faltam. No cenário 5, tempo e identidade do cliente são discutidos separadamente.

## A1. Dataset próprio
`MeuDataset` (em `lab02lib/data.py`) recebe uma tabela de uma única partição, as características e, opcionalmente, média e desvio. No treino ele aprende as duas estatísticas só com as próprias linhas. Validação e teste recebem os valores do treino e só aplicam a conta. Convenção: **desvio populacional (`ddof=0`)**, calculado em float64 e guardado em float32. Desvio zero vira 1,0.

Por que isso importa: se a média fosse calculada com a validação ou o teste, a escala do treino já conteria informação desses conjuntos. Essa é a forma 1 de vazamento, que reproduzo na Parte C.

Cada exemplo devolve `X` float32 com forma `[11]` e `y` float32 escalar. Num lote de tamanho B as formas são `[B, 11]` e `[B]`.

In [ ]:
from lab02lib.data import criar_datasets, criar_loaders, tabela_parametros, verificar_datasets

ds_treino, ds_validacao, ds_teste = criar_datasets(particoes, FEATURES)
print('Tamanhos: treino', len(ds_treino), '| validação', len(ds_validacao), '| teste', len(ds_teste))
print('\nDocstrings do MeuDataset:')
for nome in ['__init__', '__len__', '__getitem__']:
    print(f'- {nome}:', ' '.join(getattr(MeuDataset, nome).__doc__.split()))

### A2. Três DataLoaders
O treino embaralha para que cada época veja os exemplos em outra ordem. Isso muda a ordem dentro do treino, não a separação entre as partições. Validação e teste não embaralham e **não descartam nada**: a ordem das previsões precisa continuar alinhada à dos rótulos, e `obter_escores` devolve os dois juntos. As justificativas completas estão na tabela abaixo.

In [ ]:
loader_treino, loader_validacao, loader_teste = criar_loaders(ds_treino, ds_validacao, ds_teste, SEED)
loaders = {'treino': loader_treino, 'validacao': loader_validacao, 'teste': loader_teste}
assert loader_validacao.dataset is ds_validacao and loader_teste.dataset is ds_teste
a2_tabela = tabela_parametros(loaders)
a2_tabela.to_csv(RESULTADOS / 'a2_loaders.csv', index=False)
display(a2_tabela)
a1_estatisticas = verificar_datasets(ds_treino, ds_validacao, ds_teste, loaders, FEATURES)
a1_estatisticas.to_csv(RESULTADOS / 'a1_estatisticas.csv', index=False)
display(a1_estatisticas.round(4))

In [ ]:
e = a1_estatisticas.set_index('conjunto')
display(Markdown(f'''**A1 — como ler.** No treino a maior média absoluta de uma coluna é {e.max_abs_media['treino']:.1e}, praticamente zero, porque as estatísticas saíram dele. Na validação ela é {e.max_abs_media['validacao']:.3f} e no teste {e.max_abs_media['teste']:.3f}. Isso é esperado: validação e teste foram normalizados com a média e o desvio **do treino**, então não precisam ter média zero. O desvio médio por coluna é {e.desvio_medio['validacao']:.3f} na validação e {e.desvio_medio['teste']:.3f} no teste. Se eles tivessem média exatamente zero, eu desconfiaria de que usaram as próprias estatísticas.

**A2 — tabela de decisões.** Os valores abaixo vêm dos carregadores criados.

| Conjunto | batch_size | shuffle | drop_last | num_workers | Efeito da escolha |
|---|---|---|---|---|---|
| treino | {loader_treino.batch_size} | {a2_tabela.shuffle[0]} | {loader_treino.drop_last} | {loader_treino.num_workers} | Lote 64 dá {len(loader_treino)} atualizações por época com {len(ds_treino)} linhas: um meio-termo entre o lote 1 (muitas atualizações ruidosas) e o 512 (poucas atualizações), como a A3 mostra. Embaralhar, com gerador semeado, evita a mesma ordem em toda época e continua reproduzível. Não descarto o último lote (sobra {len(ds_treino) % 64} linha): a rede de referência não tem BatchNorm. |
| validação | {loader_validacao.batch_size} | {a2_tabela.shuffle[1]} | {loader_validacao.drop_last} | {loader_validacao.num_workers} | Sem gradientes, o lote só afeta memória, então cabe a validação inteira ({len(ds_validacao)} linhas) em um lote. Sem embaralhar, a ordem dos escores fica igual à dos rótulos. Nenhuma observação é descartada. |
| teste | {loader_teste.batch_size} | {a2_tabela.shuffle[2]} | {loader_teste.drop_last} | {loader_teste.num_workers} | Mesmo raciocínio da validação ({len(ds_teste)} linhas). O teste só é percorrido na avaliação final. |

`num_workers=0` nos três: os dados já estão em tensores na memória, então não há leitura de disco para esconder. Processos auxiliares custam para abrir e para trocar dados. Eu **não medi** workers maiores que 0, então não afirmo que ajudam ou atrapalham; só escolhi o caminho que não depende de criar processos.'''))

## A3. Tempo de uma época
Compara `batch_size` 1, 8, 64 e 512 com a rede **sem BatchNorm**. Uma medição é uma época completa de `treinar_epoca` (previsão, perda, retropropagação e passo do otimizador) cronometrada com `time.perf_counter()`.

Para a comparação ser justa:
- o estado inicial é guardado uma vez e cada medição reconstrói o modelo a partir dele, com um **Adam novo**;
- cada medição usa um DataLoader novo (o `loader_treino` da Parte D não é tocado);
- há uma época de aquecimento por tamanho, fora da medição;
- são três repetições por tamanho, em rodízio entre os tamanhos, para que a oscilação da máquina não caia sempre no mesmo lote;
- nenhuma medida é descartada.

In [ ]:
from lab02lib.timing import medir_tempos, resumir_tempos, grafico_tempos, descrever_hardware
import json

hardware = descrever_hardware()
tempos = medir_tempos(ds_treino, [1, 8, 64, 512], repeticoes=3, semente=SEED, dispositivo=DISPOSITIVO)
a3_resumo = resumir_tempos(tempos)
pd.DataFrame(tempos).to_csv(RESULTADOS / 'a3_tempos.csv', index=False)
a3_resumo.to_csv(RESULTADOS / 'a3_resumo.csv', index=False)
(RESULTADOS / 'a3_hardware.json').write_text(json.dumps(hardware, indent=2, ensure_ascii=False), encoding='utf-8')
display(pd.DataFrame(tempos).round(5))
display(a3_resumo.round(6))
print(hardware)
display(grafico_tempos(tempos, hardware, RESULTADOS / 'a3_tempo_x_lote.png'))

In [ ]:
r = a3_resumo.set_index('batch_size')
razoes = [r.mediana_segundos[a] / r.mediana_segundos[b] for a, b in [(1, 8), (8, 64), (64, 512)]]
display(Markdown(f'''**Como ler.** A mediana de uma época foi {r.mediana_segundos[1]:.4f} s com lote 1, {r.mediana_segundos[8]:.4f} s com lote 8, {r.mediana_segundos[64]:.4f} s com lote 64 e {r.mediana_segundos[512]:.4f} s com lote 512. Cada degrau multiplicou o lote por 8 e dividiu o tempo por {razoes[0]:.1f}, {razoes[1]:.1f} e {razoes[2]:.1f}. {'O ganho encolheu a cada degrau, mas não chegou a zero.' if razoes[0] > razoes[1] > razoes[2] and razoes[2] > 1.2 else 'A sequência de ganhos não é monotônica ou chegou perto de um platô.'} Se o último ganho ficasse perto de 1, eu diria que o tempo achatou. Não é o que aparece nesta medição.

O custo de **um passo** sobe com o lote: {r.mediana_segundos_por_passo[1]*1000:.3f} ms, {r.mediana_segundos_por_passo[8]*1000:.3f} ms, {r.mediana_segundos_por_passo[64]*1000:.3f} ms e {r.mediana_segundos_por_passo[512]*1000:.3f} ms. Cada passo processa mais linhas, mas o custo não cresce na mesma proporção do lote. Por isso a época fica mais curta com lotes grandes: há menos passos ({r.passos[1]}, {r.passos[8]}, {r.passos[64]} e {r.passos[512]}).

Lotes diferentes também significam **quantidades diferentes de atualizações de pesos por época**. Com lote 1 o otimizador atualiza {r.passos[1]} vezes; com lote 512, {r.passos[512]}. Ir mais rápido por época não quer dizer aprender o mesmo: com poucas atualizações a rede pode precisar de mais épocas. A medição do tempo não diz nada sobre a qualidade do modelo.'''))

## A4. BatchNorm com último lote 1
`nn.BatchNorm1d` em modo `train()` calcula média e variância do lote. Com **uma** única observação não existe variância, e o PyTorch levanta `ValueError`. Isso acontece quando o número de linhas deixa resto 1 na divisão pelo lote: aqui, 65 linhas do treino e lote 64 (65 % 64 = 1).

O experimento usa só 65 linhas do treino (sorteadas com semente) e `criar_modelo(11, batchnorm=True)` com `drop_last=False`. Capturo **somente** o `ValueError` com a mensagem esperada. Qualquer outro erro continua subindo. Depois, com um **modelo novo e a mesma semente** (o primeiro já deu um passo e ficou contaminado), uso a correção `drop_last=True` com `shuffle=True`. Para mostrar qual linha foi descartada, um carregador espelho com a mesma semente devolve também o `id_linha`; ele só serve para a demonstração.

In [ ]:
from lab02lib.batchnorm import demonstrar_batchnorm

a4_log = demonstrar_batchnorm(ds_treino, RESULTADOS / 'a4_log.txt', semente=SEED)

**Como ler o A4.** O erro aparece no passo do último lote, que tem uma observação. A correção descarta esse resto, e a linha descartada **muda a cada época** porque o treino embaralha antes de cortar o lote incompleto. Por isso o custo é pequeno: uma linha por época e nunca a mesma. Sem embaralhar, a mesma linha ficaria sempre de fora.

Isso vale só para o treino. Na validação e no teste eu **não** uso `drop_last=True`: cada observação precisa ser avaliada (o `apoio.py` até recusa `drop_last=True` na avaliação). E o modelo avaliado precisa estar em `eval()`, para o BatchNorm usar as estatísticas acumuladas no treino e não as do lote. O modelo principal desta entrega não usa BatchNorm; a variante só existe aqui.

## C. Dois mecanismos de vazamento
Escolhi as formas **1** (pré-processamento aprendido com dados reservados) e **4** (duplicatas entre aprendizagem e avaliação). As duas usam dados reais (Red Wine Quality), e a 4 é uma das formas 2/4/7 que o enunciado pede.

**Isolamento do teste.** Só trabalho com `c_desenvolvimento = treino + validação` da B2. O teste principal não entra: uma checagem confere que nenhum `id_linha` dele aparece nos conjuntos `c_`. Todas as variáveis desta parte começam com `c_`, para não sobrescrever nada da Parte D, e uma checagem no fim da seção confirma isso.

**Protocolo comum (declarado antes de rodar).**
- Rede 11→16→1 da D-2, `BCEWithLogitsLoss`, Adam 1e-3, lote 64, **100 épocas fixas, sem parada antecipada**.
- Métricas na avaliação interna: **acurácia (limiar 0,5) e AUC-ROC**. Também mostro a acurácia de sempre prever a classe majoritária como referência.
- **Cinco sementes declaradas antes: 0, 1, 2, 3 e 4.** A semente define a partição interna, a inicialização e o embaralhamento. Nenhuma semente é descartada nem escolhida pelo resultado.
- Incorreto e corrigido, na mesma semente, compartilham inicialização e embaralhamento e, sempre que possível, as mesmas linhas.
- Diferença = `100 × (incorreto − corrigido)`: pontos percentuais para a acurácia e pontos de AUC×100 para a AUC-ROC.
- O desvio entre sementes usa `ddof=1`. **Não é erro-padrão**: as partições internas de sementes diferentes se sobrepõem, então os cinco resultados não são independentes. Leio média ± desvio só como espalhamento entre as 5 sementes combinadas.

### Forma 1 — pré-processamento aprendido com dados reservados
- *Informação que deveria ficar protegida:* as linhas da avaliação interna (sua média e seu desvio).
- *Onde atravessa:* no cálculo do z-score. No procedimento incorreto, média e desvio vêm de **todas** as linhas de `c_desenvolvimento`, inclusive as de avaliação.
- *Conjunto de desenvolvimento:* `c_treino`, 75% dos perfis de `c_desenvolvimento`. *Avaliação interna:* `c_avaliacao`, os outros 25%, com perfis que não aparecem no treino (`GroupShuffleSplit`, `random_state=semente`).
- *Corrigido:* estatísticas só de `c_treino`. As linhas, a inicialização e o embaralhamento são os mesmos.
- *Expectativa declarada antes de medir:* efeito **perto de zero**. Com 11 características e cerca de mil linhas, incluir 25% a mais de linhas muda pouco a média e o desvio. Não vou tentar aumentar esse efeito.

### Forma 4 — duplicatas entre aprendizagem e avaliação
- *Informação que deveria ficar protegida:* os perfis da avaliação. Linhas com as mesmas 11 medidas não deveriam estar dos dois lados.
- *Onde atravessa:* na divisão. No procedimento incorreto, a divisão é **por linha** (`train_test_split`, 25% para avaliação, `random_state=semente`), então um perfil pode aparecer no treino e na avaliação `E`.
- *Corrigido:* **o mesmo `E`**, e o treino perde toda linha cujo perfil está em `E`. Assim as linhas avaliadas são idênticas e só o vazamento muda. O preço é um treino **menor**, e esse preço também entra na comparação (mostro os dois tamanhos).
- *Estatísticas:* sempre do treino de cada variante, para isolar o mecanismo da divisão.
- *Mecanismo medido:* quantas linhas de `E` têm o perfil também no treino, e a acurácia do modelo incorreto nelas contra o resto.

In [ ]:
from lab02lib.leakage import (c_desenvolvimento_de, executar_c, resumir_c, linhas_para_checagem,
                              tabela_enunciado, mecanismo_forma4, SEMENTES)

c_antes = set(globals())   # fotografia dos nomes, para provar que a parte C só criou nomes c_*
c_desenvolvimento = c_desenvolvimento_de(particoes)
c_ids_teste = set(particoes['teste']['id_linha'])
assert c_ids_teste.isdisjoint(c_desenvolvimento['id_linha'])
print('OK: nenhum id_linha do teste principal está em c_desenvolvimento ->', len(c_desenvolvimento), 'linhas (treino + validação).')
print('Sementes predeclaradas:', SEMENTES)

c_resultado = executar_c(c_desenvolvimento, SEMENTES)
c_resumo = resumir_c(c_resultado)
c_tabela = tabela_enunciado(c_resumo)
c_mecanismo = mecanismo_forma4(c_resultado)
resultados_c = linhas_para_checagem(c_resumo)
c_resultado.to_csv(RESULTADOS / 'c_resultados.csv', index=False)
c_resumo.to_csv(RESULTADOS / 'c_resumo.csv', index=False)
c_tabela.to_csv(RESULTADOS / 'c_tabela.csv', index=False)
c_mecanismo.to_csv(RESULTADOS / 'c_mecanismo_forma4.csv', index=False)
assert len(c_resultado) == len(SEMENTES) * 2 * 2
print('OK: 5 sementes × 2 formas × 2 variantes =', len(c_resultado), 'execuções; nenhuma descartada.')

In [ ]:
display(c_resultado[['forma', 'variante', 'seed', 'n_treino', 'n_avaliacao', 'positivos_avaliacao',
                     'acuracia', 'auc_roc', 'acuracia_majoritaria']].round(4))
display(c_resumo.round(4))
print('Mecanismo da forma 4 (por semente):')
display(c_mecanismo.round(4))

In [ ]:
c_r = c_resumo.set_index(['forma', 'metrica'])
c_a1, c_a4 = c_r.loc[(1, 'acurácia (limiar 0,5)')], c_r.loc[(4, 'acurácia (limiar 0,5)')]
c_u1, c_u4 = c_r.loc[(1, 'AUC-ROC')], c_r.loc[(4, 'AUC-ROC')]
c_m = c_mecanismo
display(Markdown(f'''**Forma 1.** Com a acurácia, a diferença média entre incorreto e corrigido foi de {c_a1.diferenca_pp_media:.3f} p.p. (desvio {c_a1.diferenca_pp_desvio:.3f}; entre {c_a1.diferenca_pp_min:.3f} e {c_a1.diferenca_pp_max:.3f}). Com a AUC-ROC, {c_u1.diferenca_pp_media:.3f} ponto de AUC×100 (desvio {c_u1.diferenca_pp_desvio:.3f}; entre {c_u1.diferenca_pp_min:.3f} e {c_u1.diferenca_pp_max:.3f}), positiva em {int(c_u1.sementes_com_diferenca_positiva)} das {int(c_u1.n_sementes)} sementes. {'Era o que eu esperava: efeito perto de zero.' if abs(c_u1.diferenca_pp_media) < 0.5 and abs(c_a1.diferenca_pp_media) < 0.5 else 'O efeito foi maior do que eu esperava.'} O vazamento **existe** (as estatísticas do incorreto usam as linhas de avaliação); o que a medição mostra é quanto ele pesa nesta base. Um efeito pequeno não prova que o procedimento é seguro: só diz que, com 11 características e esta quantidade de linhas, incluir mais 25% de linhas muda pouco a média e o desvio. A correção não custa nada, então fica.

**Forma 4.** A diferença média foi de {c_a4.diferenca_pp_media:.3f} p.p. na acurácia (desvio {c_a4.diferenca_pp_desvio:.3f}; positiva em {int(c_a4.sementes_com_diferenca_positiva)} de {int(c_a4.n_sementes)} sementes) e de {c_u4.diferenca_pp_media:.3f} ponto de AUC×100 na AUC-ROC (desvio {c_u4.diferenca_pp_desvio:.3f}; positiva em {int(c_u4.sementes_com_diferenca_positiva)} de {int(c_u4.n_sementes)}). O incorreto ficou acima do corrigido {'em todas as sementes' if c_u4.sementes_com_diferenca_positiva == c_u4.n_sementes else 'em parte das sementes'}, mas a diferença é {'pequena' if abs(c_u4.diferenca_pp_media) < 3 else 'grande'}, em pontos percentuais e não em dezenas.

O mecanismo é visível: em média {100*c_m.fracao_aval_com_perfil_no_treino.mean():.1f}% das linhas de `E` ({c_m.aval_com_perfil_no_treino.min()} a {c_m.aval_com_perfil_no_treino.max()} de {c_m.n_avaliacao.iloc[0]}) têm o perfil também no treino do incorreto; no corrigido são zero em todas as sementes. O treino corrigido é menor: {c_m.n_treino_corrigido.min()} a {c_m.n_treino_corrigido.max()} linhas, contra {c_m.n_treino_incorreto.min()} a {c_m.n_treino_incorreto.max()}. Isso mistura dois efeitos na diferença (o vazamento some e o treino encolhe), e eu não os separo aqui. A acurácia do incorreto nas linhas com perfil repetido foi, em média, {c_m.acuracia_nas_compartilhadas_incorreto.mean():.4f}, contra {c_m.acuracia_nas_demais_incorreto.mean():.4f} nas demais. Em {(c_m.acuracia_nas_compartilhadas_incorreto > c_m.acuracia_nas_demais_incorreto).sum()} de {len(c_m)} sementes a acurácia foi maior nas linhas com perfil repetido. Com algumas dezenas de linhas por semente, esse contraste é ruidoso, então não o trato como prova de memorização.

Duas ressalvas. Primeiro, a acurácia de sempre prever a classe majoritária fica por volta de {100*c_resultado.acuracia_majoritaria.mean():.1f}%, e a acurácia média da rede é {100*c_resultado.acuracia.mean():.1f}%, ou seja, {100*(c_resultado.acuracia - c_resultado.acuracia_majoritaria).mean():.1f} p.p. acima dela em média. Com tão pouco espaço, a AUC-ROC (que usa o ranking dos escores) é a métrica mais informativa aqui. Segundo, com 5 sementes sobrepostas, o desvio mostra espalhamento, não incerteza estatística. Uma diferença pequena, ou de sinal inesperado, não derruba a existência do vazamento.'''))

In [ ]:
c_novos = {n for n in set(globals()) - c_antes if not n.startswith('_') and n not in ('In', 'Out', 'c_antes')}
c_fora_do_padrao = sorted(n for n in c_novos if not n.startswith('c_') and n != 'resultados_c')
assert not c_fora_do_padrao, c_fora_do_padrao
print('OK: toda variável criada na parte C começa com c_ (exceto resultados_c, exigida pela conferência final).')
print('Variáveis da parte C:', sorted(c_novos))

## D7. Rede principal com parada antecipada
Rede nova (11→16→1, ReLU), semente 42, sem nada herdado da Parte C. A primeira célula **apaga as variáveis `c_`** do notebook: se algum código da Parte D ainda dependesse de algo da C, daria erro em vez de passar sem eu perceber.

A função `treinar_com_parada` (em `lab02lib/train.py`) faz, a cada época: treina uma época, avalia a validação sem gradientes, compara com a melhor perda e decide. Há melhora quando `perda_val < melhor − min_delta`. Ao melhorar, guarda uma **cópia independente** dos pesos (`copy.deepcopy` do `state_dict`, porque uma referência continuaria mudando junto com o treino), a época e a perda. Sem melhora por `paciencia` épocas seguidas, ela para. Ao sair, **restaura** os melhores pesos no modelo.

Definidos antes: `max_epocas=100`, `paciencia=10`, `min_delta=1e-4`. A função **não recebe o carregador de teste**; a assinatura é conferida abaixo. A perda de validação escolhe a época; o teste não aparece no histórico.

In [ ]:
from lab02lib.train import assinatura_sem_teste, verificar_restauracao, grafico_perdas

# Isolamento da Parte C: nenhum objeto c_ sobrevive até aqui.
for nome in [n for n in list(globals()) if n.startswith('c_')]:
    del globals()[nome]
assert not [n for n in globals() if n.startswith('c_')]
print('OK: variáveis c_ removidas. A parte D usa só particoes, os datasets e os carregadores da B2/A.')

parametros = assinatura_sem_teste()
print('Parâmetros de treinar_com_parada:', parametros)
print('OK: a função não recebe carregador de teste.')

fixar_semente(SEED)
loader_treino.generator.manual_seed(SEED)   # o embaralhamento do treino recomeça da semente
modelo_final = criar_modelo(11).to(DISPOSITIVO)
resultado_d7 = treinar_com_parada(modelo_final, loader_treino, loader_validacao,
                                  max_epocas=100, paciencia=10, min_delta=0.0001)
historico = resultado_d7.historico
historico.to_csv(RESULTADOS / 'd7_historico.csv', index=False)
display(historico.tail(12).round(5))
display(grafico_perdas(resultado_d7, RESULTADOS / 'd7_curvas_perda.png'))

In [ ]:
prova = verificar_restauracao(modelo_final, resultado_d7, loader_validacao, DISPOSITIVO)
print('OK: todos os tensores do modelo são iguais (torch.equal) ao estado guardado na melhor época.')
print(f"OK: perda de validação recalculada = {prova['perda_recalculada']:.8f}; guardada = {resultado_d7.melhor_perda:.8f}; diferença = {prova['diferenca_absoluta']:.2e} (< 1e-6).")
print('Época de parada:', resultado_d7.epoca_parada, '| melhor época:', resultado_d7.melhor_epoca,
      '| melhor perda de validação:', round(resultado_d7.melhor_perda, 6), '| motivo:', resultado_d7.motivo)

In [ ]:
r7 = resultado_d7
partes = [f'''**Como ler.** O treino parou na época {r7.epoca_parada}; a melhor época foi a {r7.melhor_epoca}, com perda de validação {r7.melhor_perda:.4f}. Motivo da parada: **{'a paciência (10 épocas sem melhora maior que 0,0001)' if r7.motivo == 'paciencia' else 'o limite de 100 épocas'}**.''']
if r7.motivo == 'max_epocas':
    partes.append('A paciência **não** disparou: a rede ainda podia estar melhorando quando as 100 épocas acabaram, então não posso dizer que a parada antecipada agiu.')
else:
    partes.append(f'Foram {r7.epoca_parada - r7.melhor_epoca} épocas depois da melhor, e os pesos usados daqui em diante são os da época {r7.melhor_epoca}, não os da última.')
h7 = r7.historico
partes.append(f'''Na melhor época a perda de treino era {h7.perda_treino[r7.melhor_epoca - 1]:.4f} e a de validação {h7.perda_validacao[r7.melhor_epoca - 1]:.4f}. Na última época, {h7.perda_treino.iloc[-1]:.4f} e {h7.perda_validacao.iloc[-1]:.4f}. {'A validação voltou a subir enquanto o treino continuou caindo: é o sinal de que a rede começou a se ajustar ao treino além do que generaliza, e é por isso que guardo o melhor estado em vez do último.' if h7.perda_validacao.iloc[-1] > r7.melhor_perda + 1e-4 and h7.perda_treino.iloc[-1] < h7.perda_treino[r7.melhor_epoca - 1] else 'Nesta execução a validação não subiu de forma clara depois da melhor época.'}''')
partes.append('A prova de restauração acima mostra que o modelo agora **é** o da melhor época (pesos iguais e perda recalculada igual), e não uma versão posterior.')
display(Markdown('\n\n'.join(partes)))

## D4. Custos como hipóteses explícitas
**Decisão.** Quando o sistema prevê positivo, ele **seleciona** a amostra para uma análise especial. Quando prevê negativo, a amostra não é selecionada.

- **Falso positivo (FP):** selecionar como alta nota uma amostra com nota abaixo de 7. Custo: uma análise especial gasta à toa.
- **Falso negativo (FN):** deixar de selecionar uma amostra com nota pelo menos 7. Custo: uma amostra boa passa sem a análise.
- Acertos custam zero (simplificação do enunciado).

**`C_FP = 1` e `C_FN = 5`, em "pontos de esforço".** **São hipóteses do exercício.** A base não traz preços, perdas nem custos operacionais reais; escolhi os números para ter uma decisão a discutir. A razão 1:5 diz que perder uma amostra de alta nota é julgado cinco vezes pior que gastar uma análise. É uma escolha, não uma medida. Com esses custos, a fórmula do limiar dá `C_FP/(C_FP+C_FN) = 1/6 ≈ 0,167`.

In [ ]:
from lab02lib.decision import C_FP, C_FN, REGRA_EMPATE
print(f'C_FP = {C_FP} | C_FN = {C_FN} | unidade: pontos de esforço (hipotéticos) | custo de acertos = 0')
print('Limiar da fórmula C_FP/(C_FP+C_FN) =', round(C_FP / (C_FP + C_FN), 4))
print('Regra de desempate (fixada antes de ver os escores):', REGRA_EMPATE)

## D5 e D6. Escolha na validação e fórmula
Com o melhor modelo restaurado, obtenho os escores da **validação** (`obter_escores`). Para cada limiar candidato, classifico como positivo quando `escore >= limiar` e calculo `custo = C_FP × FP + C_FN × FN`.

- **Candidatos:** todos os escores únicos da validação, mais um valor logo acima do máximo (`np.nextafter`) que representa "tudo negativo". O menor escore já representa "tudo positivo". A busca é **exata sobre os escores únicos, não uma grade**. Os candidatos são valores de escore, isto é, a borda superior de cada intervalo de limiares que dão as mesmas previsões.
- **Desempate** (fixado antes de ver os dados): entre limiares com o mesmo custo mínimo, o mais próximo de 0,5; se ainda empatar, o menor.
- Comparo com 0,5 **na mesma validação**, com a fórmula `C_FP/(C_FP+C_FN)` e com os casos "tudo positivo" e "tudo negativo".

**Aviso de uso da validação.** A mesma validação já decidiu a parada (D7) e agora decide o limiar. Os números desta seção são evidência de desenvolvimento, **não** a estimativa independente final. Essa vem só do teste.

In [ ]:
from lab02lib.decision import (buscar_limiar, comparar_regras, grafico_custo, calibracao,
                               congelar, verificar_congelamento, prever)

y_val, p_val = obter_escores(modelo_final, loader_validacao, DISPOSITIVO)
assert loader_validacao.dataset is ds_validacao
assert np.array_equal(y_val, ds_validacao.y.numpy().astype(int)), 'rótulos desalinhados dos escores'
print('OK: escores e rótulos da validação alinhados;', len(y_val), 'observações,', int(y_val.sum()), 'positivos.')

busca = buscar_limiar(y_val, p_val)
busca.tabela.to_csv(RESULTADOS / 'd5_busca_limiar.csv', index=False)
d5_comparacao = comparar_regras(y_val, p_val, busca)
d5_comparacao.to_csv(RESULTADOS / 'd5_comparacao.csv', index=False)
limiar_escolhido = busca.limiar
print('Candidatos avaliados:', len(busca.tabela), '| limiares com o custo mínimo:', busca.n_empatados)
display(d5_comparacao.round(4))
display(grafico_custo(busca, C_FP / (C_FP + C_FN), RESULTADOS / 'd5_curva_custo.png'))

In [ ]:
d6_tabela, d6_brier, d6_fig = calibracao(y_val, p_val, RESULTADOS / 'd6_calibracao.png', n_bins=5)
d6_tabela.to_csv(RESULTADOS / 'd6_calibracao.csv', index=False)
display(d6_tabela.round(4))
display(d6_fig)
print('Brier score na validação:', round(d6_brier, 4))

In [ ]:
b2 = pd.read_csv(RESULTADOS / 'b2_auditoria.csv').set_index('conjunto')
cmp_ = d5_comparacao.set_index('regra')
esc, p05, frm = cmp_.loc['escolhido na busca'], cmp_.loc['padrão 0,5'], cmp_.loc['fórmula C_FP/(C_FP+C_FN)']
t_formula = C_FP / (C_FP + C_FN)
prev_val = b2.proporcao_positiva['validacao']
brier_ref = prev_val * (1 - prev_val)
d6_gap = d6_tabela.iloc[(d6_tabela.escore_medio - d6_tabela.fracao_positivos).abs().argmax()]
display(Markdown(f'''**D5 — como ler.** O limiar escolhido na validação foi **{limiar_escolhido:.4f}**, com custo total {esc.custo_total:.0f} ({esc.custo_por_obs:.4f} por observação; {int(esc.fp)} FP e {int(esc.fn)} FN). Com 0,5 na mesma validação, o custo total foi {p05.custo_total:.0f} ({p05.custo_por_obs:.4f} por observação; {int(p05.fp)} FP e {int(p05.fn)} FN). {'O limiar escolhido custou menos que 0,5.' if esc.custo_total < p05.custo_total else 'O limiar escolhido custou o mesmo que 0,5.' if esc.custo_total == p05.custo_total else 'O limiar escolhido custou mais que 0,5, o que não deveria acontecer numa busca exata.'} Para referência: prever tudo positivo custa {cmp_.custo_total['tudo positivo']:.0f} e tudo negativo, {cmp_.custo_total['tudo negativo']:.0f}. {busca.n_empatados} limiar(es) empataram no custo mínimo; a regra de desempate escolheu o mais próximo de 0,5.

Como o limiar foi escolhido **nesta** validação, o custo dela é otimista: o limiar se ajustou a ela. A avaliação honesta só vem no teste.

**D6 — busca contra fórmula.** A fórmula dá {t_formula:.4f}; a busca deu {limiar_escolhido:.4f}, {'bem acima' if limiar_escolhido > 2 * t_formula else 'acima' if limiar_escolhido > t_formula else 'abaixo ou igual'}. A fórmula supõe escores que sejam probabilidades da população, custos constantes e acertos de custo zero. Eu não garanto nada disso. Razões para diferir:
- **Poucos positivos.** A validação tem {int(b2.positivos['validacao'])} positivos em {int(b2.linhas['validacao'])} linhas (proporção {100*prev_val:.2f}%, contra {100*b2.proporcao_positiva['treino']:.2f}% no treino). A curva de custo é uma escada irregular: um único FN a mais ou a menos pesa {C_FN} pontos, e o mínimo pode mudar de lugar com poucas observações.
- **Escores sem garantia de calibração.** Sigmoide de um logit não é probabilidade calibrada. A rede foi parada pela perda de validação, não calibrada. O Brier score na validação foi {d6_brier:.4f} (para comparar: um escore constante igual à prevalência da validação daria {brier_ref:.4f}). O diagrama de calibração usa 5 bins por quantis (cerca de {len(y_val)//5} observações cada; com 10 bins ficariam só alguns positivos por bin) e **apoia a discussão, não prova calibração**. O bin com maior distância entre escore médio e fração observada foi o {int(d6_gap.bin)}º (escore médio {d6_gap.escore_medio:.3f}, fração observada de positivos {d6_gap.fracao_positivos:.3f}); onde o escore supera a fração observada, a fórmula, que confia no escore, corta mais baixo do que a busca.
- **A busca é exata sobre os escores únicos**, não uma grade grosseira, então a diferença para a fórmula não vem de resolução.
- **Sem peso de classe nem reamostragem** no treino (decisão D-11). Sem peso, a perda quase não pune errar a classe rara, e os escores dos positivos tendem a ficar baixos. Isso é compatível com um corte abaixo de 0,5, mas eu não testei essa explicação.

Não ajusto o modelo para as duas respostas coincidirem, e não olhei o teste.'''))

### Congelando a decisão
Antes de abrir o teste, gravo em `resultados/decisao_congelada.json`: limiar, regra de desempate, `C_FP` e `C_FN`, lista de características, média e desvio do treino, arquitetura, melhor época, **SHA-256 dos pesos restaurados** e semente. Só depois disso `decisao_congelada` vira `True`. A avaliação final confere o hash de novo e se recusa a rodar se algo mudou.

In [ ]:
decisao = congelar(RESULTADOS / 'decisao_congelada.json', modelo_final, limiar_escolhido, ds_treino,
                   FEATURES, resultado_d7.melhor_epoca, SEED)
decisao_congelada = True
print('Decisão congelada em resultados/decisao_congelada.json')
print('limiar =', decisao['limiar'], '| melhor época =', decisao['melhor_epoca'])
print('sha256 dos pesos =', decisao['sha256_pesos_restaurados'])

## D1, D2 e D3. Avaliação final
**Só agora o teste é aberto**, com o modelo restaurado e a decisão congelada. Os escores do teste são obtidos **uma única vez** (um objeto conta as chamadas e recusa a segunda). Avalio as duas regras já fixadas, o limiar 0,5 e o limiar escolhido na validação, e o classificador trivial.

Convenções:
- Matriz de confusão: **linhas = classe real, colunas = classe prevista**, ordem `[0, 1]`.
- AUC-ROC e AUC-PR saem dos **escores**, não das classes depois do corte; por isso são iguais nas duas linhas da rede. AUC-PR é a área trapezoidal de `auc(recall, precision)` sobre `precision_recall_curve`. A `average_precision_score` aparece separada, como **AP**, e não é a mesma coisa.
- Se nenhuma previsão for positiva, a precisão é 0 (`zero_division=0`).
- Trivial: a classe majoritária **do treino**, prevista para todo o teste. Um preditor constante tem um único ponto de operação e nenhum ranking; por isso AUC-ROC e AUC-PR ficam como "não reportada".
- Custo por observação: `(C_FP×FP + C_FN×FN)/n`, em pontos de esforço hipotéticos, para as três linhas.

Depois desta seção eu não volto a mexer em nada.

In [ ]:
from lab02lib.evaluate import EscoresUmaVez, avaliar_teste, matriz_rotulada, grafico_roc_pr

assert decisao_congelada
verificar_congelamento(RESULTADOS / 'decisao_congelada.json', modelo_final, limiar_escolhido)
print('OK: o hash dos pesos e o limiar são os do arquivo congelado.')

obtentor = EscoresUmaVez(modelo_final, loader_teste, DISPOSITIVO)
y_teste, p_teste = obtentor.obter()
assert obtentor.chamadas == 1
assert np.array_equal(y_teste, ds_teste.y.numpy().astype(int))
print('OK: escores do teste obtidos exatamente', obtentor.chamadas, 'vez; rótulos alinhados com o dataset de teste.')

d_tabela, d_matrizes, areas = avaliar_teste(y_teste, p_teste, ds_treino.y.numpy().astype(int), limiar_escolhido)
avaliacao_final = d_tabela
d_tabela.to_csv(RESULTADOS / 'd_tabela_final.csv', index=False)
d_matrizes.to_csv(RESULTADOS / 'd_matrizes.csv', index=False)
print(f"Teste: {len(y_teste)} linhas, {int(y_teste.sum())} positivos (proporção positiva = {areas['prevalencia_teste']:.4f}).")
print(f"AUC-ROC = {areas['auc_roc']:.4f} | AUC-PR trapezoidal = {areas['auc_pr']:.4f} | AP = {areas['ap']:.4f}")
display(d_tabela.round(4))
for regra in ['Rede: padrão', 'Rede: escolhido na validação']:
    display(matriz_rotulada(d_matrizes, regra))

In [ ]:
display(grafico_roc_pr(y_teste, p_teste, limiar_escolhido, areas, RESULTADOS / 'd_curvas_roc_pr.png'))

In [ ]:
t = d_tabela.set_index('regra')
triv, r05, rch = t.loc['Trivial do treino'], t.loc['Rede: padrão'], t.loc['Rede: escolhido na validação']
display(Markdown(f'''**Como ler.** No teste, {int(y_teste.sum())} de {len(y_teste)} linhas são positivas ({100*areas['prevalencia_teste']:.2f}%). O trivial (sempre classe 0) acerta {100*triv.acuracia:.2f}%, mas tem revocação {triv.revocacao:.2f}: **não encontra nenhuma amostra de nota alta**, que era o objetivo. Por isso a acurácia dele é alta e inútil. Custo dele: {triv.custo_por_obs:.4f} por observação.

A rede com 0,5 tem acurácia {100*r05.acuracia:.2f}%, precisão {r05.precisao:.3f}, revocação {r05.revocacao:.3f}, F1 {r05.f1:.3f} e custo {r05.custo_por_obs:.4f} por observação ({int(r05.fp)} FP, {int(r05.fn)} FN). Com o limiar escolhido na validação ({limiar_escolhido:.4f}): acurácia {100*rch.acuracia:.2f}%, precisão {rch.precisao:.3f}, revocação {rch.revocacao:.3f}, F1 {rch.f1:.3f} e custo {rch.custo_por_obs:.4f} por observação ({int(rch.fp)} FP, {int(rch.fn)} FN). {'O limiar menor encontra mais positivos, com mais falsos positivos, e o custo hipotético caiu.' if rch.revocacao > r05.revocacao and rch.custo_por_obs < r05.custo_por_obs else 'No teste, o limiar escolhido na validação não reduziu o custo em relação a 0,5.' if rch.custo_por_obs >= r05.custo_por_obs else 'O custo mudou, mas o padrão de revocação não foi o esperado.'} O enunciado não exige acurácia mínima, e ela sozinha esconde o que importa aqui: o custo e a revocação contam melhor a história.

AUC-ROC {areas['auc_roc']:.3f} e AUC-PR {areas['auc_pr']:.3f} (AP {areas['ap']:.3f}) vêm dos escores e valem para as duas regras. A linha de referência da PR é a prevalência do teste ({areas['prevalencia_teste']:.3f}); a curva da rede fica {'acima' if areas['auc_pr'] > areas['prevalencia_teste'] else 'abaixo'} dela. A diagonal da ROC marca o aleatório, de área 0,5.

Esta é uma única avaliação, com {int(y_teste.sum())} positivos no teste. Diferenças pequenas entre as duas regras ficam dentro do que {int(y_teste.sum())} observações positivas permitem distinguir.'''))

## D8. Recomendação final
O parágrafo para quem decide está no `README.md` (seção D8) e usa só os números gerados acima. Os custos são **hipóteses do exercício**, o conjunto avaliado é o teste de Red Wine Quality separado na B2, e nada aqui vale para produtores, regiões ou períodos que a base não permite avaliar.

## Conferência da entrega
Esta última célula é a checagem estrutural do template e deve imprimir uma lista vazia.

In [ ]:
pendencias = []
for nome, valor in [('particoes', particoes), ('ds_treino', ds_treino),
    ('loader_treino', loader_treino), ('historico', historico),
    ('limiar_escolhido', limiar_escolhido), ('avaliacao_final', avaliacao_final)]:
    if valor is None:
        pendencias.append(nome)
if not tempos: pendencias.append('tempos A3')
if len(resultados_c) < 2: pendencias.append('dois resultados C')
print('Pendências estruturais:', pendencias)
print('Esta checagem não corrige a análise nem substitui o checklist do enunciado.')